# Causal LM Summarization: Small Qwen / DeepSeek LoRA

Notebook nay chay them causal LM LoRA trong cung repo. Tat ca model trong cell mac dinh deu <3B tham so. Mac dinh chay Qwen3-1.7B LoRA; Qwen2.5-1.5B va DeepSeek-R1-Distill-Qwen-1.5B tat mac dinh.


In [1]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
from zipfile import ZipFile

PROJECT_NAME = 'pretrained-summarization'
REPO_URL = 'https://github.com/Anhnguyen0812/pretrained-summarization.git'
REFRESH_REPO = True
WORKING = Path('/kaggle/working')
WORKING_REPO = WORKING / PROJECT_NAME
OUTPUT_ROOT = WORKING / 'summarization_outputs'
DATA_DIR = Path('/kaggle/input/datasets/anhnguyen0812/nlp-vietnamese-sumarization')
TRAIN_FILE = DATA_DIR / 'train-00000-of-00001.parquet'
VALID_FILE = DATA_DIR / 'valid-00000-of-00001.parquet'

os.chdir(WORKING)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=None, check=True):
    print('CMD:', cmd, flush=True)
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    process = subprocess.Popen(cmd, shell=True, cwd=str(cwd) if cwd else None, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines=[]
    for line in process.stdout:
        print(line, end='', flush=True)
        lines.append(line)
    code=process.wait()
    if check and code != 0:
        raise RuntimeError(f'Command failed with exit code {code}: {cmd}\nLast lines:\n{"".join(lines[-100:])}')
    return code

def is_repo(path):
    return (path / 'pyproject.toml').exists() and (path / 'src' / 'vn_summarization').exists()

if REFRESH_REPO and WORKING_REPO.exists():
    shutil.rmtree(WORKING_REPO)
if not is_repo(WORKING_REPO):
    run(f'git clone --depth 1 {REPO_URL} {WORKING_REPO}', cwd=WORKING)
else:
    run('git pull --ff-only', cwd=WORKING_REPO, check=False)
repo = WORKING_REPO
run('git log --oneline -1', cwd=repo)

configs_dir = repo / 'configs'

def write_causal_lora_config(path, model_name, output_dir, prompt_template):
    if path.exists():
        return
    prompt_yaml = json.dumps(prompt_template, ensure_ascii=False)
    path.write_text(f'''model:
  name_or_path: {model_name}
  use_fast_tokenizer: true
  trust_remote_code: false
  cache_dir:
  padding_side: right
  torch_dtype: float16
  max_parameters: 3000000000

data:
  train_file: ../train-00000-of-00001.parquet
  valid_file: ../valid-00000-of-00001.parquet
  max_source_length: 768
  max_target_length: 128
  max_length: 896
  preprocessing_num_proc: 1
  max_train_samples: 2500
  max_eval_samples: 120
  prompt_template: {prompt_yaml}

training:
  output_dir: {output_dir}
  overwrite_output_dir: false
  seed: 42
  precision: fp16
  per_device_train_batch_size: 1
  per_device_eval_batch_size: 2
  gradient_accumulation_steps: 8
  num_train_epochs: 2
  max_steps: 300
  learning_rate: 0.0001
  weight_decay: 0.01
  warmup_ratio: 0.03
  lr_scheduler_type: cosine
  optim: adamw_torch
  gradient_checkpointing: true
  strategy: steps
  save_strategy: steps
  eval_steps: 300
  save_steps: 300
  logging_steps: 25
  save_total_limit: 2
  load_best_model_at_end: true
  metric_for_best_model: eval_loss
  greater_is_better: false
  report_to:
    - tensorboard
  save_safetensors: true

generation:
  max_new_tokens: 128
  num_beams: 1
  do_sample: false
  no_repeat_ngram_size: 3
  repetition_penalty: 1.08

lora:
  enabled: true
  r: 8
  lora_alpha: 16
  lora_dropout: 0.05
  target_modules:
    - q_proj
    - k_proj
    - v_proj
    - o_proj
    - gate_proj
    - up_proj
    - down_proj
''', encoding='utf-8')

base_prompt = 'Bạn là trợ lý tóm tắt tiếng Việt. Viết duy nhất một đoạn tóm tắt ngắn, trung lập, đủ ý chính. Giữ tên riêng, số liệu, mốc thời gian quan trọng. Không thêm thông tin ngoài văn bản, không giải thích.\n\nVăn bản:\n{article}\n\nTóm tắt:\n'
no_think_prompt = 'Bạn là trợ lý tóm tắt tiếng Việt. Viết duy nhất một đoạn tóm tắt cuối cùng, không viết suy luận, không giải thích, không dùng thẻ <think>. Giữ tên riêng, số liệu, mốc thời gian quan trọng. Không thêm thông tin ngoài văn bản.\n\nVăn bản:\n{article}\n\nTóm tắt:\n'
small_configs = [
    ('qwen3_1_7b_lora.yaml', 'Qwen/Qwen3-1.7B', 'outputs/qwen3_1_7b_lora', no_think_prompt),
    ('qwen25_1_5b_lora.yaml', 'Qwen/Qwen2.5-1.5B-Instruct', 'outputs/qwen25_1_5b_lora', base_prompt),
    ('deepseek_r1_distill_qwen_1_5b_lora.yaml', 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B', 'outputs/deepseek_r1_distill_qwen_1_5b_lora', no_think_prompt),
]
for file_name, model_name, output_dir, prompt in small_configs:
    write_causal_lora_config(configs_dir / file_name, model_name, output_dir, prompt)
print('SMALL_CAUSAL_CONFIGS', [file_name for file_name, *_ in small_configs])


CMD: git clone --depth 1 https://github.com/Anhnguyen0812/pretrained-summarization.git /kaggle/working/pretrained-summarization
Cloning into '/kaggle/working/pretrained-summarization'...
CMD: git log --oneline -1
00ddcae Fix Kaggle causal LM dependency versions
SMALL_CAUSAL_CONFIGS ['qwen3_1_7b_lora.yaml', 'qwen25_1_5b_lora.yaml', 'deepseek_r1_distill_qwen_1_5b_lora.yaml']


In [2]:
os.chdir(repo)
run(f'{sys.executable} -m pip install -q --upgrade pip', cwd=repo)
run(f'{sys.executable} -m pip install -q -e .', cwd=repo)
run(f'{sys.executable} -m pip install -q --upgrade "transformers>=4.51.0,<5" "tokenizers>=0.22.0,<=0.23.0"', cwd=repo)
run(f'{sys.executable} -m pip check', cwd=repo, check=False)
run(f'{sys.executable} -m pip show transformers tokenizers peft accelerate | sed -n "/Name: /p;/Version: /p"', cwd=repo, check=False)
run(f"{sys.executable} -c 'import tokenizers, transformers; print(\"TRANSFORMERS\", transformers.__version__); print(\"TOKENIZERS\", tokenizers.__version__)'", cwd=repo)


CMD: /usr/bin/python3 -m pip install -q --upgrade pip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 22.1 MB/s eta 0:00:00
CMD: /usr/bin/python3 -m pip install -q -e .
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompa

0

In [3]:
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
run('nvidia-smi', check=False)
print('TRAIN_FILE:', TRAIN_FILE, TRAIN_FILE.exists())
print('VALID_FILE:', VALID_FILE, VALID_FILE.exists())
if not TRAIN_FILE.exists() or not VALID_FILE.exists():
    raise FileNotFoundError('Attach Kaggle dataset first.')

import torch
NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
print('NUM_GPUS', NUM_GPUS)


CMD: nvidia-smi
Wed Jun 10 05:51:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-------------------------------

In [4]:
BASE_OVERRIDES = [
    f'data.train_file={TRAIN_FILE}',
    f'data.valid_file={VALID_FILE}',
]
RUN_CONFIGS = {}

def _override_string(items):
    return ' '.join(f'--set {item}' for item in items)

def latest_checkpoint(run_dir):
    checkpoints = sorted(run_dir.glob('checkpoint-*'), key=lambda p: int(p.name.split('-')[-1]) if p.name.split('-')[-1].isdigit() else -1)
    return checkpoints[-1] if checkpoints else None

def train_causal(run_name, config_path, overrides=None, overwrite=False):
    run_dir = OUTPUT_ROOT / run_name
    RUN_CONFIGS[run_name] = config_path
    if (run_dir / 'best' / 'adapter_config.json').exists() and not overwrite:
        print('SKIP train, adapter exists:', run_dir / 'best')
        return 0
    resume = []
    if run_dir.exists() and not overwrite:
        ckpt = latest_checkpoint(run_dir)
        if ckpt:
            print('RESUME', ckpt)
            resume = [f'training.resume_from_checkpoint={ckpt}']
        else:
            print('CLEAR incomplete run:', run_dir)
            shutil.rmtree(run_dir)
    final_overrides = BASE_OVERRIDES + [f'training.output_dir={run_dir}'] + resume + (overrides or [])
    train_cmd = f'-m vn_summarization.train_causal_lm --config {config_path} {_override_string(final_overrides)}'
    if NUM_GPUS >= 2:
        cmd = f'{sys.executable} -m accelerate.commands.launch --multi_gpu --num_processes {NUM_GPUS} --num_machines 1 --mixed_precision fp16 --dynamo_backend no {train_cmd}'
    else:
        cmd = f'{sys.executable} -u {train_cmd}'
    return run(cmd, cwd=repo)

def eval_causal(eval_name, config_path, model_path, overrides=None, overwrite=False):
    eval_dir = OUTPUT_ROOT / eval_name
    if (eval_dir / 'validation_metrics.json').exists() and not overwrite:
        print('SKIP eval:', eval_dir / 'validation_metrics.json')
        return 0
    eval_dir.mkdir(parents=True, exist_ok=True)
    final_overrides = BASE_OVERRIDES + [f'training.output_dir={eval_dir}'] + (overrides or [])
    cmd = (
        f'{sys.executable} -u -m vn_summarization.evaluate_causal_lm '
        f'--config {config_path} --model_path {model_path} '
        f'--predictions_path {eval_dir / "predictions_valid.jsonl"} '
        f'{_override_string(final_overrides)}'
    )
    return run(cmd, cwd=repo)

def summarize_results():
    return run(f'{sys.executable} -u -m vn_summarization.summarize_results --root {OUTPUT_ROOT}', cwd=repo, check=False)


## Run Flags

Mac dinh: Qwen3-1.7B va DeepSeek-R1-Distill-Qwen-1.5B LoRA profile nhanh ~3h tren Kaggle T4x2. Profile nay dung max_steps=300/model, 2500 train samples, 120 eval samples, greedy generation. Tang samples/steps neu can diem cao hon.


In [5]:
RUN_QWEN3 = True
RUN_QWEN25 = False
RUN_DEEPSEEK = True
OVERWRITE_RUNS = False
OVERWRITE_EVALS = False

COMMON_CAUSAL = [
    'training.num_train_epochs=1',
    'training.max_steps=300',
    'training.eval_steps=300',
    'training.save_steps=300',
    'training.logging_steps=25',
    'training.save_total_limit=2',
    'training.ddp_find_unused_parameters=false',
    'training.per_device_train_batch_size=1',
    'training.per_device_eval_batch_size=2',
    'training.gradient_accumulation_steps=8',
    'data.max_train_samples=2500',
    'data.max_eval_samples=120',
    'data.max_source_length=768',
    'data.max_target_length=128',
    'data.max_length=896',
    'generation.max_new_tokens=128',
    'generation.num_beams=1',
    'lora.r=8',
    'lora.lora_alpha=16',
]

EXPERIMENTS=[]
if RUN_QWEN3:
    EXPERIMENTS.append({
        'name': 'qwen3_1_7b_lora_fast3h_t4x2',
        'config': 'configs/qwen3_1_7b_lora.yaml',
        'overrides': COMMON_CAUSAL,
    })
if RUN_QWEN25:
    EXPERIMENTS.append({
        'name': 'qwen25_1_5b_lora_fast3h_t4x2',
        'config': 'configs/qwen25_1_5b_lora.yaml',
        'overrides': COMMON_CAUSAL,
    })
if RUN_DEEPSEEK:
    EXPERIMENTS.append({
        'name': 'deepseek_r1_qwen_1_5b_lora_fast3h_t4x2',
        'config': 'configs/deepseek_r1_distill_qwen_1_5b_lora.yaml',
        'overrides': COMMON_CAUSAL,
    })
print(json.dumps(EXPERIMENTS, indent=2))


[
  {
    "name": "qwen3_1_7b_lora_fast3h_t4x2",
    "config": "configs/qwen3_1_7b_lora.yaml",
    "overrides": [
      "training.num_train_epochs=1",
      "training.max_steps=300",
      "training.eval_steps=300",
      "training.save_steps=300",
      "training.logging_steps=25",
      "training.save_total_limit=2",
      "training.ddp_find_unused_parameters=false",
      "training.per_device_train_batch_size=1",
      "training.per_device_eval_batch_size=2",
      "training.gradient_accumulation_steps=8",
      "data.max_train_samples=2500",
      "data.max_eval_samples=120",
      "data.max_source_length=768",
      "data.max_target_length=128",
      "data.max_length=896",
      "generation.max_new_tokens=128",
      "generation.num_beams=1",
      "lora.r=8",
      "lora.lora_alpha=16"
    ]
  },
  {
    "name": "deepseek_r1_qwen_1_5b_lora_fast3h_t4x2",
    "config": "configs/deepseek_r1_distill_qwen_1_5b_lora.yaml",
    "overrides": [
      "training.num_train_epochs=1",
      

## Train And Evaluate

Sau train, notebook evaluate ROUGE bang generation tren validation. Evaluation co the ton thoi gian vi causal LM generate cham hon seq2seq.


In [6]:
for exp in EXPERIMENTS:
    print('\n' + '='*100)
    print('TRAIN', exp['name'])
    train_causal(exp['name'], exp['config'], exp['overrides'], overwrite=OVERWRITE_RUNS)
    model_path = OUTPUT_ROOT / exp['name'] / 'best'
    print('EVAL', model_path)
    eval_causal(exp['name'] + '_eval', exp['config'], model_path, exp['overrides'], overwrite=OVERWRITE_EVALS)
    summarize_results()



TRAIN qwen3_1_7b_lora_fast3h_t4x2
CMD: /usr/bin/python3 -m accelerate.commands.launch --multi_gpu --num_processes 2 --num_machines 1 --mixed_precision fp16 --dynamo_backend no -m vn_summarization.train_causal_lm --config configs/qwen3_1_7b_lora.yaml --set data.train_file=/kaggle/input/datasets/anhnguyen0812/nlp-vietnamese-sumarization/train-00000-of-00001.parquet --set data.valid_file=/kaggle/input/datasets/anhnguyen0812/nlp-vietnamese-sumarization/valid-00000-of-00001.parquet --set training.output_dir=/kaggle/working/summarization_outputs/qwen3_1_7b_lora_fast3h_t4x2 --set training.num_train_epochs=1 --set training.max_steps=300 --set training.eval_steps=300 --set training.save_steps=300 --set training.logging_steps=25 --set training.save_total_limit=2 --set training.ddp_find_unused_parameters=false --set training.per_device_train_batch_size=1 --set training.per_device_eval_batch_size=2 --set training.gradient_accumulation_steps=8 --set data.max_train_samples=2500 --set data.max_eval_

## Summarize And Export


In [7]:
summarize_results()
zip_path = WORKING / 'causal_lm_results.zip'
if zip_path.exists():
    zip_path.unlink()
keep_suffixes = {'.json', '.jsonl', '.csv', '.md', '.txt'}
files = [p for p in OUTPUT_ROOT.rglob('*') if p.is_file() and p.suffix in keep_suffixes]
with ZipFile(zip_path, 'w') as zf:
    for file in files:
        zf.write(file, file.relative_to(WORKING).as_posix())
print('ZIP', zip_path)
for file in sorted(files)[:80]:
    print(file)


CMD: /usr/bin/python3 -u -m vn_summarization.summarize_results --root /kaggle/working/summarization_outputs
# Result Summary

| run | model | lora | rouge1 | rouge2 | rougeL | gen_len | loss |
| --- | --- | --- | --- | --- | --- | --- | --- |
| qwen3_1_7b_lora_fast3h_t4x2_eval |  |  | 61.8926 | 28.0069 | 33.9245 | 78.1333 |  |
| deepseek_r1_qwen_1_5b_lora_fast3h_t4x2_eval |  |  | 45.9363 | 18.1163 | 27.3552 | 66.6167 |  |
| deepseek_r1_qwen_1_5b_lora_fast3h_t4x2 | deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B | True |  |  |  |  | 1.5886 |
| qwen3_1_7b_lora_fast3h_t4x2 | Qwen/Qwen3-1.7B | True |  |  |  |  | 0.6347 |

CSV: /kaggle/working/summarization_outputs/summary_results.csv
Best: /kaggle/working/summarization_outputs/best_run.json
ZIP /kaggle/working/causal_lm_results.zip
/kaggle/working/summarization_outputs/best_run.json
/kaggle/working/summarization_outputs/deepseek_r1_qwen_1_5b_lora_fast3h_t4x2/all_results.json
/kaggle/working/summarization_outputs/deepseek_r1_qwen_1_5b_lora_fast3h